# 📘 Notebook 6 — Apprendimento NON supervisionato: K-Means e Clustering

Finora abbiamo sempre avuto un **target** (la "risposta giusta"): si chiama **apprendimento supervisionato**.

Ora vediamo l'**apprendimento NON supervisionato**: dati **senza etichette**, e il modello deve scoprire **da solo** la struttura nei dati.

## 🎯 Cosa imparerai qui
1. Differenza tra **supervisionato** e **non supervisionato**
2. Cos'è il **clustering** e a cosa serve
3. L'algoritmo **K-Means** spiegato passo passo
4. Come **scegliere k** (numero di cluster) con il metodo del gomito
5. Esempio pratico: **segmentazione clienti**
6. Limiti del K-Means e alternative (DBSCAN)

## 🤔 Quando usare il clustering?
- **Segmentazione clienti**: raggruppare clienti simili per marketing mirato
- **Compressione di immagini**: ridurre i colori dell'immagine
- **Rilevamento anomalie**: trovare punti che non appartengono a nessun gruppo
- **Esplorazione dati**: capire la struttura di un dataset sconosciuto

## 1️⃣ Supervisionato vs Non Supervisionato

| | Supervisionato | Non Supervisionato |
|---|---|---|
| Dati | X + y (etichette) | Solo X |
| Obiettivo | Prevedere y per nuovi X | Scoprire struttura |
| Esempi | Regressione, classificazione | Clustering, riduzione dimensionalità |
| Valutazione | Confronto con etichette vere | Più difficile (non c'è verità) |


![kmeans](Media/06/kmeans.png)



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

# Generiamo 3 "nuvole" di punti (in realtà sappiamo le classi, ma facciamo finta di non saperle)
X, y_vero = make_blobs(
    n_samples=300,
    centers=3,         # 3 gruppi
    cluster_std=1.0,   # quanto sono "sparsi"
    random_state=42
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# A sinistra: come VEDIAMO i dati (senza etichette)
axes[0].scatter(X[:, 0], X[:, 1], c='gray', s=30)
axes[0].set_title('Cosa VEDE il computer (dati grezzi)')
axes[0].grid(alpha=0.3)

# A destra: la verità nascosta (3 gruppi reali)
axes[1].scatter(X[:, 0], X[:, 1], c=y_vero, cmap='viridis', s=30)
axes[1].set_title('Verità nascosta (3 gruppi)')
axes[1].grid(alpha=0.3)

plt.show()

# 💡 Il clustering deve scoprire i gruppi a destra avendo solo i dati a sinistra!

## 2️⃣ K-Means: l'algoritmo più famoso

### Idea (semplicissima!)
1. **Decidi quanti cluster** vuoi: $k$ (es: $k = 3$)
2. **Posiziona $k$ punti casuali** come "centri" provvisori (centroidi)
3. **Assegna ogni punto** al centroide più vicino
4. **Sposta ogni centroide** alla media dei suoi punti
5. **Ripeti** 3 e 4 finché niente cambia più


![clustering](Media/06/clustering.png)

Vediamolo in azione, passo passo!

In [ ]:
# Implementazione MANUALE del K-Means per capire i passi
np.random.seed(7)
k = 3

# 1. Inizializziamo k centroidi a caso
centroidi = X[np.random.choice(len(X), k, replace=False)].copy()

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for iter_num in range(6):
    # 2. Assegniamo ogni punto al centroide più vicino
    distanze = np.sqrt(((X[:, np.newaxis] - centroidi) ** 2).sum(axis=2))
    assegnazioni = distanze.argmin(axis=1)
    
    # Disegniamo
    ax = axes[iter_num]
    ax.scatter(X[:, 0], X[:, 1], c=assegnazioni, cmap='viridis', s=30, alpha=0.6)
    ax.scatter(centroidi[:, 0], centroidi[:, 1], c='red', marker='X', s=200, edgecolor='k', linewidth=2)
    ax.set_title(f'Iterazione {iter_num}')
    ax.grid(alpha=0.3)
    
    # 3. Aggiorniamo i centroidi: media dei punti assegnati
    nuovi_centroidi = np.array([X[assegnazioni == i].mean(axis=0) for i in range(k)])
    
    if np.allclose(centroidi, nuovi_centroidi):
        break
    centroidi = nuovi_centroidi

plt.suptitle('K-Means in azione: i centroidi (X rossi) si spostano fino a stabilizzarsi', fontsize=13)
plt.tight_layout()
plt.show()

## 3️⃣ K-Means con scikit-learn

Nella pratica si usa la versione di sklearn (più veloce e robusta).

In [ ]:
from sklearn.cluster import KMeans

# Creiamo e addestriamo K-Means
kmeans = KMeans(
    n_clusters=3,    # k = 3
    n_init=10,       # ripete 10 volte con inizializzazioni diverse e tiene il migliore
    random_state=42
)
kmeans.fit(X)

# I cluster assegnati
etichette = kmeans.labels_
centri = kmeans.cluster_centers_

plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c=etichette, cmap='viridis', s=30, alpha=0.7)
plt.scatter(centri[:, 0], centri[:, 1], c='red', marker='X', s=200, edgecolor='k', linewidth=2, label='Centroidi')
plt.title('K-Means con scikit-learn')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# .predict() per nuovi punti
nuovi_punti = np.array([[0, 0], [-5, -5], [5, 5]])
print("Previsioni cluster per nuovi punti:", kmeans.predict(nuovi_punti))

## 4️⃣ Come scegliere $k$? Il metodo del gomito

Il problema: **noi diciamo a K-Means quanti cluster cercare**. Ma se non sappiamo?

**Idea**: provare diversi $k$ e misurare la **somma delle distanze al quadrato** (chiamata **inerzia**) di ogni punto al suo centroide.
- $k$ piccolo → inerzia alta (cluster troppo grandi e "stiracchiati")
- $k$ grande → inerzia bassa, ma stiamo "barando" (col limite $k = N$ ogni punto è il proprio cluster)

Si cerca il **"gomito"** del grafico: il punto in cui aggiungere un cluster non riduce molto l'inerzia.
![gomito](Media/06/gomito.png)



In [ ]:
# Proviamo k da 1 a 10
inerzie = []
k_range = range(1, 11)
for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    inerzie.append(km.inertia_)

plt.figure(figsize=(9, 5))
plt.plot(k_range, inerzie, 'o-', linewidth=2, markersize=10)
plt.xlabel('Numero di cluster (k)')
plt.ylabel('Inerzia (somma distanze²)')
plt.title('Metodo del gomito')
plt.axvline(3, color='red', linestyle='--', label='Il "gomito" è qui (k=3)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# 💡 Si vede che dopo k=3 la curva si appiattisce → 3 è la scelta giusta.

### Alternativa: il **Silhouette Score**

Misura quanto i punti sono **vicini al loro cluster** e **lontani dagli altri**.
- Va da -1 (male) a +1 (ottimo)
- Si sceglie il $k$ con silhouette più alto

In [ ]:
from sklearn.metrics import silhouette_score

sil_scores = []
k_range = range(2, 11)   # silhouette non si calcola per k=1
for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    sil_scores.append(silhouette_score(X, km.labels_))

plt.figure(figsize=(9, 5))
plt.plot(list(k_range), sil_scores, 's-', linewidth=2, markersize=10, color='green')
plt.xlabel('Numero di cluster (k)')
plt.ylabel('Silhouette score')
plt.title('Silhouette: quale k è migliore?')
plt.grid(alpha=0.3)
plt.show()

k_best = list(k_range)[np.argmax(sil_scores)]
print(f"k migliore secondo silhouette: {k_best}")

## 5️⃣ Esempio pratico: segmentazione clienti

Caso aziendale tipico: **un negozio vuole capire i suoi clienti** per fare marketing mirato.

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Generiamo un dataset clienti realistico
np.random.seed(0)
n = 500

# 4 "tipi" di clienti che vogliamo scoprire:
# 1. Giovani che spendono poco
# 2. Adulti con reddito alto e spesa alta (target premium)
# 3. Anziani parsimoniosi
# 4. Spendaccioni di mezza età

tipo = np.random.choice([0, 1, 2, 3], size=n, p=[0.3, 0.2, 0.3, 0.2])
eta = np.where(tipo==0, np.random.normal(25, 5, n),
        np.where(tipo==1, np.random.normal(40, 8, n),
        np.where(tipo==2, np.random.normal(65, 7, n),
                          np.random.normal(45, 5, n)))).clip(18, 80)
reddito = np.where(tipo==0, np.random.normal(20, 5, n),
        np.where(tipo==1, np.random.normal(80, 15, n),
        np.where(tipo==2, np.random.normal(35, 8, n),
                          np.random.normal(60, 12, n)))).clip(10, 200)   # in migliaia di euro/anno
spesa_annua = np.where(tipo==0, np.random.normal(500, 200, n),
        np.where(tipo==1, np.random.normal(4000, 800, n),
        np.where(tipo==2, np.random.normal(800, 300, n),
                          np.random.normal(3000, 600, n)))).clip(100, 8000)

clienti = pd.DataFrame({'eta': eta.round(0), 'reddito_k': reddito.round(1), 'spesa': spesa_annua.round(0)})
clienti.head()

In [ ]:
# IMPORTANTE: prima di K-Means, scaliamo i dati!
# Le scale di età (10-80), reddito (10-200), spesa (100-8000) sono molto diverse:
# senza scaling, la spesa dominerebbe il calcolo delle distanze.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(clienti)

# Troviamo il k migliore
inerzie = []
for k in range(1, 11):
    inerzie.append(KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_scaled).inertia_)

plt.figure(figsize=(9, 4))
plt.plot(range(1, 11), inerzie, 'o-')
plt.title('Metodo del gomito sui clienti')
plt.xlabel('k'); plt.ylabel('inerzia'); plt.grid(alpha=0.3)
plt.show()

# Scegliamo k=4
kmeans = KMeans(n_clusters=4, n_init=10, random_state=42)
clienti['cluster'] = kmeans.fit_predict(X_scaled)

# Cosa caratterizza ogni cluster?
profili = clienti.groupby('cluster').mean().round(1)
print("📊 Profilo di ciascun cluster:")
print(profili)

In [ ]:
# Visualizziamo i cluster
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(clienti['eta'], clienti['spesa'], c=clienti['cluster'], cmap='tab10', s=30, alpha=0.7)
axes[0].set_xlabel('Età')
axes[0].set_ylabel('Spesa annua (€)')
axes[0].set_title('Cluster nello spazio Età × Spesa')

axes[1].scatter(clienti['reddito_k'], clienti['spesa'], c=clienti['cluster'], cmap='tab10', s=30, alpha=0.7)
axes[1].set_xlabel('Reddito (k€)')
axes[1].set_ylabel('Spesa annua (€)')
axes[1].set_title('Cluster nello spazio Reddito × Spesa')

plt.tight_layout()
plt.show()

# 💡 Ora possiamo dare nomi "di business" ai cluster, ad esempio:
# - Cluster con eta bassa, spesa bassa  → "Giovani esploratori"
# - Cluster con reddito alto, spesa alta → "VIP / Premium"
# - ecc.
# Il marketing può poi creare campagne diverse per ogni segmento!

## 6️⃣ Limiti del K-Means

K-Means è semplice e veloce, ma ha **forti assunzioni**:
1. **Cluster sferici** di dimensione simile
2. **Numero $k$** da decidere a priori
3. **Sensibile alle scale** delle variabili → scaling obbligatorio!
4. **Sensibile agli outlier**
5. **Non gestisce forme strane** (es: cluster a forma di luna)

![funziona](Media/06/funziona.png)


In [ ]:
from sklearn.datasets import make_moons
from sklearn.cluster import DBSCAN

# Generiamo dati a forma di MEZZALUNA
X_moon, _ = make_moons(n_samples=300, noise=0.08, random_state=42)

# Confrontiamo K-Means e DBSCAN (un altro algoritmo di clustering)
kmeans_moon = KMeans(n_clusters=2, n_init=10, random_state=42).fit(X_moon)
dbscan_moon = DBSCAN(eps=0.2, min_samples=5).fit(X_moon)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(X_moon[:, 0], X_moon[:, 1], c=kmeans_moon.labels_, cmap='viridis', s=30)
axes[0].set_title('K-Means → FALLISCE su forme non sferiche!')
axes[1].scatter(X_moon[:, 0], X_moon[:, 1], c=dbscan_moon.labels_, cmap='viridis', s=30)
axes[1].set_title('DBSCAN → riconosce le mezzelune')
plt.show()

# 💡 DBSCAN è basato sulla DENSITÀ, non sulle distanze ai centroidi.
# Funziona bene per cluster di forma arbitraria e individua anche outlier (etichetta -1).

## 7️⃣ Applicazione divertente: compressione di un'immagine

Un'immagine ha **migliaia di colori** diversi. Usando K-Means possiamo trovare i $k$ colori "più rappresentativi" e mantenere solo quelli.

In [ ]:
# Creiamo un'immagine sintetica con un gradiente colorato
h, w = 200, 200
img = np.zeros((h, w, 3))
for i in range(h):
    for j in range(w):
        img[i, j] = [i / h, j / w, ((i+j) / (h+w))]

# Trasformiamo l'immagine in una lista di pixel
pixels = img.reshape(-1, 3)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(img); axes[0].set_title('Originale (migliaia di colori)'); axes[0].axis('off')

for i, k in enumerate([2, 5, 16]):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(pixels)
    # Sostituiamo ogni pixel con il colore del suo centroide
    nuovi_pixels = km.cluster_centers_[km.labels_]
    img_compressa = nuovi_pixels.reshape(h, w, 3)
    axes[i+1].imshow(img_compressa)
    axes[i+1].set_title(f'Solo {k} colori')
    axes[i+1].axis('off')

plt.tight_layout(); plt.show()

# 💡 Con solo 16 colori l'immagine è quasi indistinguibile dall'originale!
# Questa è una semplice forma di COMPRESSIONE.

## 🎓 Riepilogo del Notebook 6
![mondo](Media/06/modno.png)

### K-Means
- ✅ Veloce, semplice, scala bene su grandi dataset
- ❌ Devi scegliere $k$, assume cluster sferici, sensibile a scaling e outlier

### Quando usarlo
- Segmentazione clienti / utenti
- Compressione (colori, audio…)
- Esplorazione preliminare di dati nuovi
- Quando i cluster sono "compatti" e ben separati

### Quando NON usarlo
- Cluster di forme strane → usa DBSCAN o clustering gerarchico
- Tante dimensioni → meglio prima ridurre con PCA (lo vedremo!)
- Cluster molto sbilanciati come dimensione

### 🔑 Concetti chiave
| Concetto | Significato |
|---|---|
| Centroide | Centro di un cluster |
| Inerzia | Somma delle distanze² al centroide più vicino |
| Metodo del gomito | Per scegliere $k$ |
| Silhouette | Metrica alternativa per la qualità |
| Scaling | Obbligatorio prima del K-Means |

👉 **Prossimo notebook (07)**: la **riduzione della dimensionalità** con la **PCA** — come comprimere dati con tante variabili senza perdere informazione, e come visualizzare in 2D dati che hanno 100 dimensioni!